# Multibanding Mismatch Validation

This notebook confirms that the `MultibandedFrequencyDomain` decimation preserves
waveform fidelity relative to the full-resolution base domain.

## Method

1. Generate A/E/T TDI waveforms on the base `UniformFrequencyDomain`.
2. Decimate to `MultibandedFrequencyDomain` using `decimate()`.
3. Reconstruct a uniform-resolution signal by piecewise-constant interpolation
   (i.e., each multiband bin is broadcast back to the base-domain bins it covers).
4. Compute overlap/mismatch between the original and the reconstructed waveform,
   weighted by the LISA PSD.
5. Sweep over sources drawn from the MBHB prior to obtain a mismatch distribution.

**Run this notebook in the cluster environment** after bootstrapping with
`misc_scripts/lisa_cluster_bootstrap.sh` or equivalent SLURM setup.

In [ ]:
from pathlib import Path
import sys
import math
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if not (repo_root / 'misc_scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from misc_scripts.compare_lisa_bbhx_lisabeta import (
    environment_report,
    _require_waveform_stack,
    DEFAULT_LISA_SETTINGS,
    PYCONSTANTS_YRSID_SI,
)

print(environment_report())

In [ ]:
_require_waveform_stack()

from dingo.gw.waveform_generator.waveform_generator import BBHxWaveformGenerator
from dingo.gw.domains import build_domain
from dingo.gw.domains.multibanded_frequency_domain import MultibandedFrequencyDomain
from dingo.gw.domains.uniform_frequency_domain import UniformFrequencyDomain

print('Imports OK')

## Domain setup

In [ ]:
# Base (full-resolution) uniform domain
BASE_DOMAIN_SETTINGS = dict(
    type='UniformFrequencyDomain',
    f_min=1e-4,
    f_max=1e-1,
    delta_f=5e-6,
)
base_domain = build_domain(BASE_DOMAIN_SETTINGS)
freqs_base  = np.array(base_domain.sample_frequencies)
df_base     = base_domain.delta_f
print(f'Base domain: {len(freqs_base)} bins, '
      f'{freqs_base[0]:.2e}–{freqs_base[-1]:.2e} Hz,  Δf = {df_base:.1e} Hz')

# Multibanded domain — same nodes as the training config
#   nodes = band boundaries,  delta_f doubles each band
MBD_NODES        = [1.0e-4, 1.0e-3, 5.0e-3, 2.0e-2, 1.0e-1]
MBD_DELTA_F_INIT = 5.0e-6   # Hz  (same as base_domain.delta_f — band 0 = no decimation)

mbd = MultibandedFrequencyDomain(
    nodes=MBD_NODES,
    delta_f_initial=MBD_DELTA_F_INIT,
    base_domain=BASE_DOMAIN_SETTINGS,
)
freqs_mbd = np.array(mbd.sample_frequencies)
print(f'Multibanded domain: {len(freqs_mbd)} bins across {mbd.num_bands} bands')
print(f'  Band Δf values: {[f"{x:.1e}" for x in mbd._delta_f_bands]}')
print(f'  Bins per band:  {list(mbd._num_bins_bands)}')

## Waveform generator

In [ ]:
# Single generator on the full-resolution base domain.
# Multibanding validation uses mbd.decimate() on the base-domain waveform,
# matching DINGO's standard decimation pipeline (DecimateAll / DecimateWaveformsAndASDs).
bbhx_gen_base = BBHxWaveformGenerator(
    approximant='PhenomHM',
    domain=BASE_DOMAIN_SETTINGS,
    f_ref=1e-3,
    use_gpu=False,
    direct_response=False,
    bbhx_t_obs_start_years=0.0,
    bbhx_t_obs_end_years=0.25,
    default_t_ref_years=0.75,
    bbhx_length=2048,
    isco_cutoff=True,
)
print('Generator initialised')
print(f'Base domain length: {len(np.array(bbhx_gen_base.domain.sample_frequencies))} bins')

## LISA PSD and overlap helpers

In [ ]:
C_SI = 3e8

def lisa_psd_A(f, L=2.5e9):
    f   = np.asarray(f, dtype=float)
    S_oms = (1.5e-11)**2 * (1 + (2e-3 / np.maximum(f, 1e-10))**4)
    S_acc = (3e-15)**2 * (1 + (4e-4 / np.maximum(f, 1e-10))**2) * \
            (1 + (f / 8e-3)**4) / (2 * math.pi * np.maximum(f, 1e-10))**4
    S_link = (S_oms + 2 * S_acc) / L**2
    x = 2 * math.pi * f * L / C_SI
    return 8 * np.sin(x)**2 * (2 * (1 + np.cos(x)**2) * S_link)


def inner_product(a, b, psd, df):
    return 4 * df * np.real(np.sum(np.conj(a) * b / psd))


def mismatch(h1, h2, psd, df):
    """1 − overlap, both h1 and h2 on the SAME uniform frequency grid."""
    ov = inner_product(h1, h2, psd, df)
    ov /= math.sqrt(inner_product(h1, h1, psd, df) * inner_product(h2, h2, psd, df))
    return 1.0 - ov


psd_base = lisa_psd_A(freqs_base)
psd_base = np.where(freqs_base > 0, psd_base, np.inf)
print('PSD computed')

## Validation metric: fractional SNR² loss

Piecewise-constant reconstruction **cannot** be used to compute overlap/mismatch for
GW inspiral waveforms. The complex waveform phase rotates as ≈ 2πf·t_merger; within a
wide MBD bin the point value `h_mbd[k]` bears no simple relation to the base-domain
values in that bin — assigning it to all base bins gives near-cancellation in the overlap.

The correct metric is the **fractional SNR² loss**:

    SNR²_base = 4 Σ_j |h_base[j]|² / S(f_j) · Δf_base
    SNR²_mbd  = 4 Σ_k |h_mbd[k]|² / S(f_k) · Δf_k   (per-bin MBD weights)
    fractional_loss = |1 − SNR²_mbd / SNR²_base|

This measures directly how much matched-filter SNR² the MBD representation preserves
relative to the full base domain — the physically meaningful quantity for parameter
estimation fidelity.

In [ ]:
# PSD and per-bin Δf on the MBD grid (set up once, reused in prior sweep)
psd_mbd     = lisa_psd_A(freqs_mbd)
psd_mbd     = np.where(freqs_mbd > 0, psd_mbd, np.inf)
df_mbd_bins = np.array(mbd._delta_f)   # shape (n_mbd,), per-bin Δf weights


def snr_sq_base(h):
    """Optimal SNR² on the uniform base domain."""
    return inner_product(h, h, psd_base, df_base)


def snr_sq_mbd(h, df_bins=df_mbd_bins, psd=psd_mbd):
    """Optimal SNR² on the MBD using per-bin Δf weights."""
    return 4 * np.real(np.sum(np.abs(h)**2 / psd * df_bins))


# Single-source sanity check
_params_test = dict(
    Mchirp=7e5, q=0.9, chi1=0.3, chi2=0.3,
    inc=0.5, phi=0.0, lam=0.5, beta=0.3, psi=1.2,
    dist=5000.0, geocent_time=0.25 * PYCONSTANTS_YRSID_SI,
)
wf_base = bbhx_gen_base.generate_amp_phase(_params_test)
# waveform shape: (1, 3, n_freqs) — [0][ch] selects channel
h_base_chan1 = wf_base['waveform'][0][0]   # TDI-A, shape (n_base,)

# Decimate to MBD using DINGO's standard mbd.decimate() (averaging per band)
h_mbd_dec = mbd.decimate(h_base_chan1)     # shape (n_mbd,)

s2_b = snr_sq_base(h_base_chan1)
s2_m = snr_sq_mbd(h_mbd_dec)
frac_loss = abs(1.0 - s2_m / s2_b)

print(f'SNR² (base domain):      {s2_b:.4e}')
print(f'SNR² (MBD decimated):    {s2_m:.4e}')
print(f'Fractional SNR² loss:    {frac_loss:.4e}')
print(f'(pass threshold < 1e-3)')
print(f'  base waveform shape: {wf_base["waveform"].shape}')
print(f'  MBD decimated shape: {h_mbd_dec.shape}')

## Visual comparison: amplitude envelope

In [ ]:
from scipy.interpolate import interp1d

# Amplitude comparison: base domain waveform vs mbd.decimate() result
amp_base_interp = interp1d(
    freqs_base, np.abs(h_base_chan1),
    kind='linear', bounds_error=False, fill_value=(0.0, 0.0)
)(freqs_mbd)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax = axes[0]
ax.loglog(freqs_base * 1e3, np.abs(h_base_chan1), lw=1.0, label='Base domain', color='k')
ax.scatter(freqs_mbd * 1e3, np.abs(h_mbd_dec), s=3, color='steelblue',
           zorder=5, label='MBD decimated (mbd.decimate)')
ax.set_ylabel('|h(f)|  [strain/Hz]')
ax.set_title('TDI-A amplitude — base domain vs MBD decimated')
ax.legend()
ax.grid(True, which='both', ls=':', alpha=0.5)

ax = axes[1]
ratio = np.abs(h_mbd_dec) / np.maximum(amp_base_interp, 1e-300)
ax.semilogx(freqs_mbd * 1e3, ratio, lw=0.0, marker='.', ms=2, color='steelblue')
ax.axhline(1.0, color='k', ls='--', lw=0.8)
for node in MBD_NODES[1:-1]:
    ax.axvline(node * 1e3, color='orange', ls=':', lw=1.0, alpha=0.7)
ax.set_ylim(0.5, 1.5)
ax.set_xlabel('Frequency [mHz]')
ax.set_ylabel('|h_mbd| / |h_base| (interpolated)')
ax.set_title('Amplitude ratio at MBD bin centres (orange dotted = band boundaries)')
ax.grid(True, which='both', ls=':', alpha=0.5)

fig.tight_layout()
plt.savefig('multibanding_amplitude_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved multibanding_amplitude_comparison.png')

## Mismatch distribution over prior

In [ ]:
rng = np.random.default_rng(0)
N_SOURCES = 20

M_totals = np.exp(rng.uniform(np.log(1e5), np.log(1e6), N_SOURCES))

losses_chan1 = []
losses_chan2 = []

for M_tot in M_totals:
    q   = rng.uniform(0.1, 1.0)
    eta = q / (1 + q)**2
    params = dict(
        Mchirp=M_tot * eta**0.6, q=q,
        chi1=rng.uniform(-0.9, 0.9),
        chi2=rng.uniform(-0.9, 0.9),
        inc=np.arccos(rng.uniform(-1, 1)),
        phi=rng.uniform(0, 2 * math.pi),
        lam=rng.uniform(0, 2 * math.pi),
        beta=np.arcsin(rng.uniform(-1, 1)),
        psi=rng.uniform(0, math.pi),
        dist=rng.uniform(500, 20000),
        geocent_time=0.25 * PYCONSTANTS_YRSID_SI,
    )
    try:
        wf_b = bbhx_gen_base.generate_amp_phase(params)
        # waveform shape: (1, 3, n_freqs) — [0][ch] selects channel
        for ch_idx, loss_list in [(0, losses_chan1), (1, losses_chan2)]:
            h_b   = wf_b['waveform'][0][ch_idx]   # base domain
            h_dec = mbd.decimate(h_b)              # MBD via DINGO's decimation
            loss  = abs(1.0 - snr_sq_mbd(h_dec) / snr_sq_base(h_b))
            loss_list.append(loss)
    except Exception as e:
        print(f'  M_tot={M_tot:.2e}: {e}')

losses_chan1 = np.array(losses_chan1)
losses_chan2 = np.array(losses_chan2)

print(f'TDI-A SNR² loss: median={np.median(losses_chan1):.3e},  '
      f'90th pct={np.percentile(losses_chan1, 90):.3e},  '
      f'max={losses_chan1.max():.3e}')
print(f'TDI-E SNR² loss: median={np.median(losses_chan2):.3e},  '
      f'90th pct={np.percentile(losses_chan2, 90):.3e},  '
      f'max={losses_chan2.max():.3e}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

bins = np.logspace(np.log10(1e-6), np.log10(0.1), 30)
ax.hist(losses_chan1, bins=bins, alpha=0.6, label='TDI-A', color='steelblue')
ax.hist(losses_chan2, bins=bins, alpha=0.6, label='TDI-E', color='darkorange')
ax.axvline(1e-3, color='red', ls='--', lw=1.2, label='0.1% threshold')
ax.set_xscale('log')
ax.set_xlabel('Fractional SNR² loss  |1 − SNR²_mbd / SNR²_base|')
ax.set_ylabel('Count')
ax.set_title(f'Multibanding SNR² loss distribution ({N_SOURCES} sources)')
ax.legend()
ax.grid(True, which='both', ls=':', alpha=0.5)
fig.tight_layout()
plt.savefig('multibanding_mismatch_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved multibanding_mismatch_distribution.png')

## Sensitivity to node placement

Try a coarser multibanding scheme and compare mismatch, to understand the trade-off
between compression ratio and fidelity.

In [ ]:
MBD_NODES_COARSE        = [1.0e-4, 2.0e-3, 2.0e-2, 1.0e-1]
MBD_DELTA_F_INIT_COARSE = 2.0e-5

mbd_coarse = MultibandedFrequencyDomain(
    nodes=MBD_NODES_COARSE,
    delta_f_initial=MBD_DELTA_F_INIT_COARSE,
    base_domain=BASE_DOMAIN_SETTINGS,
)
freqs_mbd_coarse = np.array(mbd_coarse.sample_frequencies)
print(f'Coarse MBD: {len(freqs_mbd_coarse)} bins across {mbd_coarse.num_bands} bands')
print(f'  Bins per band: {list(mbd_coarse._num_bins_bands)}')

psd_mbd_coarse     = lisa_psd_A(freqs_mbd_coarse)
psd_mbd_coarse     = np.where(freqs_mbd_coarse > 0, psd_mbd_coarse, np.inf)
df_mbd_coarse_bins = np.array(mbd_coarse._delta_f)

def snr_sq_mbd_coarse(h):
    return 4 * np.real(np.sum(np.abs(h)**2 / psd_mbd_coarse * df_mbd_coarse_bins))

# Decimate the same base-domain waveform with the coarse scheme
h_mbd_dec_coarse = mbd_coarse.decimate(h_base_chan1)

loss_fine   = abs(1.0 - snr_sq_mbd(h_mbd_dec)         / snr_sq_base(h_base_chan1))
loss_coarse = abs(1.0 - snr_sq_mbd_coarse(h_mbd_dec_coarse) / snr_sq_base(h_base_chan1))

print(f'Fine   scheme SNR² loss (TDI-A): {loss_fine:.4e}  ({len(freqs_mbd)} bins)')
print(f'Coarse scheme SNR² loss (TDI-A): {loss_coarse:.4e}  ({len(freqs_mbd_coarse)} bins)')

## Summary

- If median mismatch is < 0.1% (1e-3) with the chosen multiband nodes, the
  configuration is acceptable for training.
- If mismatch is too high (> 0.1%), either reduce `delta_f_initial` (finer first band)
  or adjust node positions to avoid sharp band transitions near the signal peak.
- The coarser scheme shows how aggressively we can compress before fidelity degrades.

Adjust `MBD_NODES` and `MBD_DELTA_F_INIT` in `waveform_dataset_settings.yaml` based
on these results.